In [ ]:
import nltk
import evaluate
import numpy as np
from datasets import load_dataset
from transformers import T5Tokenizer, DataCollatorForSeq2Seq
from transformers import T5ForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

In [ ]:
import pandas as pd

In [ ]:
df1 = pd.read_csv('DATA/paleo_filtered3_s.csv')

In [ ]:
print(df1.head())

            author          published_at            updated_at  like_count  \
0    Daniel Warton  2022-10-19T08:22:38Z  2022-10-19T08:22:38Z           1   
1  Christian Oscar  2022-06-24T13:43:26Z  2022-06-24T13:43:26Z          11   
2     Oreo Cookies  2022-02-02T04:47:17Z  2022-02-02T04:47:17Z           2   
3   HACKERS GAMING  2021-10-06T01:02:07Z  2021-10-06T01:02:07Z           2   
4   Daniel Padilla  2021-10-01T07:53:15Z  2021-10-01T07:53:15Z           2   

                                                text  \
0  I’ve been doing keto and finding the restricti...   
1  while keto did help me lose weight it always g...   
2  Alls I know is greens tear up my stomach. I no...   
3  If you want to lose fat, I suggest trying the ...   
4  Keto works but honestly I felt disgusting from...   

                                         video  \
0  https://www.youtube.com/watch?v=R9mN-d2CDUU   
1  https://www.youtube.com/watch?v=R9mN-d2CDUU   
2  https://www.youtube.com/watch?v=R9mN-d2CD

In [ ]:
#model_name='facebook/bart-large-cnn'
#model_name = "./results/checkpoint-22500"
#model_name='google/flan-t5-base'
model_name='./my_awesome_billsum_model2'
#model_name = "./dialogue-summary-training-1702378992"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

In [ ]:
for i, index in enumerate(example_indices):
    dialogue = df1['text'][index]
    summary = df1['summary'][index]

    prompt = f"""
Comment:

{dialogue}

What was going on?
"""

    inputs = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs["input_ids"],
            max_new_tokens=100,
        )[0],
        skip_special_tokens=True
    )

    print(dash_line)
    print('Example ', i + 1)
    print(dash_line)
    print(f'INPUT PROMPT:\n{prompt}')
    print(dash_line)
    print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
    print(dash_line)
    print(f'MODEL GENERATION - ZERO SHOT:\n{output}\n')

---------------------------------------------------------------------------------------------------
Example  1
---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Comment:

while keto did help me lose weight it always give me theses headaches, u say its normal but my doctor say otherwise. he put me into this diet plant which is agoge and i never felt better when on a diet. keto never is the best for everyone

What was going on?

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
The author experienced headaches on the keto diet despite it helping them lose weight. Their doctor advised against keto and suggested the Agoge diet plan which led to better results and a more positive experience. The author concludes that keto isn't the best option for everyone.

-------------------------------------------------------------------------------------------------

In [ ]:
def make_prompt(example_indices_full, example_index_to_summarize):
    prompt = ''
    for index in example_indices_full:
        dialogue = df1['text'][index]
        summary = df1['summary'][index]

        # The stop sequence '{summary}\n\n\n' is important for FLAN-T5. Other models may have their own preferred stop sequence.
        prompt += f"""
Comment:

{dialogue}

What was going on?
{summary}


"""

    dialogue = df1['text'][example_index_to_summarize]

    prompt += f"""
Comment:

{dialogue}

What was going on?
"""

    return prompt

In [ ]:
example_indices_full = [1]
example_index_to_summarize = 2

one_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)

print(one_shot_prompt)


Comment:

while keto did help me lose weight it always give me theses headaches, u say its normal but my doctor say otherwise. he put me into this diet plant which is agoge and i never felt better when on a diet. keto never is the best for everyone

What was going on?
The author experienced headaches on the keto diet despite it helping them lose weight. Their doctor advised against keto and suggested the Agoge diet plan which led to better results and a more positive experience. The author concludes that keto isn't the best option for everyone.



Comment:

Alls I know is greens tear up my stomach. I no longer suffering from horrible gas, belching or bloating. IF has helped. I don&#39;t feel hungry, no problem saying no to pastries.  my problem candy and pizza so Once a month treat.  I seem to be eating a combo of both.  Paleo n keto. Grains are not good for me but different types of beans are. Guess you just find your body&#39;s fine line. I have been told NAFLD but numbers good when

In [ ]:
summary = df1['summary'][example_index_to_summarize]

inputs = tokenizer(one_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ONE SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
Terry found that eliminating greens from their diet helped with gas belching and bloating. They combined aspects of Paleo and keto avoiding grains but incorporating beans. They found exercise including uphill walking and playing with a Nerf ball to be beneficial. Terry encourages others to experiment and not feel discouraged if they stumble as perfection is not the goal.[Image of A person walking uphill][Image of A person playing with a Nerf ball]

---------------------------------------------------------------------------------------------------
MODEL GENERATION - ONE SHOT:
I'm eating greens and candy.


In [ ]:
example_indices_full = [1, 3, 4]
example_index_to_summarize = 2

few_shot_prompt = make_prompt(example_indices_full, example_index_to_summarize)

print(few_shot_prompt)


Comment:

while keto did help me lose weight it always give me theses headaches, u say its normal but my doctor say otherwise. he put me into this diet plant which is agoge and i never felt better when on a diet. keto never is the best for everyone

What was going on?
The author experienced headaches on the keto diet despite it helping them lose weight. Their doctor advised against keto and suggested the Agoge diet plan which led to better results and a more positive experience. The author concludes that keto isn't the best option for everyone.



Comment:

If you want to lose fat, I suggest trying the Agoge diet. I lost lots of fat and felt great all the way.

What was going on?
**I lost a lot of fat and felt fantastic on the Agoge diet. If you want to achieve similar results give it a try!**[Image of Agoge diet]



Comment:

Keto works but honestly I felt disgusting from the start. Then I tried the Agoge diet plan and it also worked, but I didn&#39;t feel bad at all, so I am stickin

In [ ]:
summary = df1['summary'][example_index_to_summarize]
generation_config = GenerationConfig(max_new_tokens=10, do_sample=True, temperature=0.5)


inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs["input_ids"],
        generation_config=generation_config,
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - FEW SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
Terry found that eliminating greens from their diet helped with gas belching and bloating. They combined aspects of Paleo and keto avoiding grains but incorporating beans. They found exercise including uphill walking and playing with a Nerf ball to be beneficial. Terry encourages others to experiment and not feel discouraged if they stumble as perfection is not the goal.[Image of A person walking uphill][Image of A person playing with a Nerf ball]

---------------------------------------------------------------------------------------------------
MODEL GENERATION - FEW SHOT:
I am eating greens. I am not sick of greens. I have been told NAFLD but numbers good when tested. I have been told nafLD but numbers good when tested?
